# USITC DataWeb 关税与供应链研究：端到端 Walkthrough

这份 Notebook 的目标不是再造一套分析代码，而是把项目主线讲清楚：

**Config → DataWeb / FRED 原始快照 → tidy trade panel → 校验后的指标 → 图表与研究输出**

所有正式计算都放在 `src/usitc_pipeline/`，这里仅负责调用、检查和解释。这样 CLI、Notebook、报告与 PPT 会使用同一套数字，不会各算一遍。

## 0. 先明确六条研究边界

1. DataWeb 的 `country of origin` 是海关认定的原产地，**不等于所有增加值都来自该经济体**。
2. 本项目默认使用 **imports for consumption**；不要和 general imports 混在一起。
3. `calculated duty / customs value` 是实际税负率（realized duty rate），**不是法定税率表本身**。
4. 数量只能在同一 HTS 产品、同一 quantity unit 下比较；不能把台、公斤和升直接加总。
5. 2026 年若只到 8 月，就是 YTD，不能和完整年度当作同口径增长。
6. 政策事件前后的变化是描述性证据。没有识别策略时，不把相关时间变化写成关税的因果效应。

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

# 同时兼容从项目根目录和 notebooks/ 目录启动 Jupyter 的情形。
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from usitc_pipeline.analysis import (
    build_domestic_production_screen,
    build_trade_panel,
    validate_trade_panel,
)
from usitc_pipeline.config import load_config
from usitc_pipeline.normalize import parse_dataweb_response

print(f'Project root: {PROJECT_ROOT}')

## 1. 配置是研究设计的唯一入口

年份、HTS 产品、重点原产地、政策节点和 FRED proxy 映射都应进入配置文件，而不是散落在绘图脚本里。这里先读配置，确认本次运行究竟分析了什么。

In [ ]:
config = load_config(PROJECT_ROOT / 'config' / 'research.toml')
config

## 2. Acquisition：保存不可变的原始快照

正式抓取由 CLI 执行：

```powershell
python run.py acquire
```

API token 只从环境变量 `DATAWEB_TOKEN` 读取；不写进 Notebook、配置、日志或输出文件。每个原始文件都应进入 run manifest，并记录 SHA-256、查询条件和获取时间。

In [ ]:
raw_dir = PROJECT_ROOT / 'data' / 'raw' / 'dataweb'
raw_files = sorted(raw_dir.glob('*.json'))
print(f'Found {len(raw_files)} cached DataWeb snapshot(s).')
for path in raw_files[:10]:
    print(' -', path.name)

## 3. Normalize：把 DataWeb 的三个 tab 合并成一张 tidy 表

DataWeb 常把 Customs Value、First Unit of Quantity、Calculated Duties 放在三个 table 中。parser 负责：

- 展平嵌套表头；
- 保留 HTS code 的前导零；
- 统一 period、frequency 和 measure 名称；
- 在 `period + HTS + origin` 键上合并三个 measure；
- 保留 source file 作为审计线索；
- 遇到重复键时停止，而不是悄悄重复计数。

In [ ]:
tidy_parts = []
for path in raw_files:
    with path.open('r', encoding='utf-8') as handle:
        payload = json.load(handle)
    tidy_parts.append(parse_dataweb_response(payload, source_file=str(path)))

if tidy_parts:
    tidy = pd.concat(tidy_parts, ignore_index=True)
    display(tidy.head())
    print(f'{len(tidy):,} normalized origin-product-period rows')
else:
    tidy = pd.DataFrame()
    print('No raw snapshot yet. Run `python run.py acquire` or `python run.py import-legacy`.')

## 4. Analysis：从来源地记录生成同口径指标

`build_trade_panel` 只接收 DataFrame，不访问网络，也不读隐含的全局变量。它在产品—期间层面生成 world total、value share、quantity share、unit value 和 realized duty rate。

份额回答的是“美国从哪里进口”，不是“产品的全部增加值在哪里产生”。例如标记为越南原产的电脑仍可能包含来自中国、韩国或其他经济体的零部件。

In [ ]:
if not tidy.empty:
    panel = build_trade_panel(tidy)
    validation = validate_trade_panel(panel, strict=True)
    display(panel.head())
    print('Validation:', validation)
else:
    panel = pd.DataFrame()

## 5. FRED domestic-production screen：官方序列，人工映射

FRED 文件中的 index value 来自 Federal Reserve industrial-production series；但是 HTS 847130 → NAICS 3341 / IPG3341A 是我们为了方向性核验而做的**人工宽口径匹配**。它只能检验“国内产出回升是否具有大类层面的可能性”，不能证明某个 HTS 产品已经 reshored。

年度变化计算为 `(compare index / base index - 1) × 100`。source value、计算结果和 mapping quality 必须分列保存。

In [ ]:
fred_path = PROJECT_ROOT / 'data' / 'processed' / 'fred_series.csv'
mapping_path = PROJECT_ROOT / 'config' / 'hts_naics_proxy.csv'

if fred_path.exists() and mapping_path.exists():
    fred = pd.read_csv(fred_path, parse_dates=['date'])
    mapping = pd.read_csv(mapping_path, dtype={'hts_code': 'string', 'naics': 'string'})
    production_screen = build_domestic_production_screen(
        fred, mapping, base_year=2024, compare_year=2025
    )
    display(production_screen)
else:
    print('FRED cache is not available yet; run the acquisition/analysis pipeline first.')

## 6. Charts 与交付物

统一运行：

```powershell
python run.py all
```

图表模块只读取 processed tables；图表旁必须有同名 chart-data CSV。报告和 PPT 也只读取这些表与图，不能在排版脚本中硬编码研究数字。

In [ ]:
chart_dir = PROJECT_ROOT / 'outputs' / 'charts'
chart_files = sorted(chart_dir.glob('*.png'))
for chart in chart_files:
    audit_csv = chart.with_suffix('.csv')
    print(f'{chart.name}: chart data = {audit_csv.name}, exists = {audit_csv.exists()}')

## 7. 输出前的最小 QA 清单

- origin value shares 是否约等于 100%；
- origin 汇总是否与 world total 一致；
- 同一产品—期间是否混用了 quantity unit；
- 是否把 YTD 与 full year 直接比较；
- realized duty rate 的 customs-value denominator 是否为正；
- HTS code 长度和前导零是否保留；
- 是否存在重复 product—origin—period 键；
- FRED series ID、retrieval date 和人工 mapping quality 是否保留；
- 每张图是否配套 chart-data CSV；
- run manifest 是否记录输入 hash 与配置快照。

通过这些检查后，我们才能讨论 sourcing pattern、产品异质性和政策时间节点；仍然要把“测量”“归因”“因果”分开。